In [0]:
#%run ./transform_data ----- A décommmenter pour lancer les notebooks séparements

# dim_language

Référentiel des langues du modèle, alimenté depuis `parameters_languages`.

Cette table porte le filtre RLS : c'est la **seule** table sur laquelle un rôle de
sécurité est écrit. Elle propage ensuite son filtre à toutes les tables de
traduction via les relations 1 -> * sur `language`.

## Rapprochement avec USERCULTURE()

`USERCULTURE()` renvoie une culture complète (`fr-FR`, `cs-CZ`, ...) dont la
partie langue suit la norme **ISO 639-1**. La colonne `code` de
`parameters_languages` ne la suit pas partout : elle porte le code **pays** pour
l'ukrainien (`UA` au lieu de `uk`) et le tchèque (`CZ` au lieu de `cs`).

Un rapprochement direct sur `code` basculerait donc silencieusement ces deux
langues sur le repli anglais. On construit une colonne dédiée `culture_code`
via un mapping explicite, jamais dérivée de `code`.

> Toute nouvelle langue ajoutée par le front doit être ajoutée à `CULTURE_MAP`.
> À défaut elle tombera sur le repli anglais (dégradé, mais pas cassant).

In [0]:
# Mapping code métier -> code langue ISO 639-1 attendu de USERCULTURE()
CULTURE_MAP = {
    "FR": "fr",
    "EN": "en",
    "PL": "pl",
    "UA": "uk",   # ukrainien : ISO 639-1 = uk, la source porte le code pays UA
    "RO": "ro",
    "CZ": "cs",   # tchèque   : ISO 639-1 = cs, la source porte le code pays CZ
}

# Langue de repli quand la culture de l'utilisateur n'est pas dans le référentiel
FALLBACK_LANGUAGE_CODE = "en"

culture_expr = F.create_map([F.lit(x) for x in sum(CULTURE_MAP.items(), ())])

In [0]:
dim_language = (
    spark.table(f"{source_catalog}.parameters_languages")
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_language").alias("language"),
        F.col("code"),
        F.col("name").alias("language_label"),
        culture_expr[F.col("code")].alias("culture_code"),
    )
)

## Garde-fou : détecter une langue non mappée

Si une langue est ajoutée dans `parameters_languages` sans être déclarée dans
`CULTURE_MAP`, `culture_code` sera `null` et cette langue deviendra
inatteignable par le RLS. On le rend visible au rafraîchissement plutôt que de
le découvrir en production.

In [0]:
langues_non_mappees = dim_language.filter(F.col("culture_code").isNull())

if langues_non_mappees.count() > 0:
    print("ATTENTION : langues absentes de CULTURE_MAP (inatteignables par le RLS) :")
    display(langues_non_mappees.select("language", "code", "language_label"))
else:
    if verbose_mode == 'debug':
        print("Toutes les langues actives sont mappées vers une culture.")
        display(dim_language)

In [0]:
# La langue de repli doit exister, sinon le garde-fou du rôle RLS ne protège rien
if dim_language.filter(F.col("culture_code") == FALLBACK_LANGUAGE_CODE).count() == 0:
    raise ValueError(
        f"La langue de repli '{FALLBACK_LANGUAGE_CODE}' est absente de dim_language : "
        "le rôle RLS renverrait un rapport vide pour toute culture non reconnue."
    )

Import dim_language

In [0]:
current_process = "dim_language"

In [0]:
target_dim_language = current_catalog + "." + current_schema + "." + current_process
print(target_dim_language)

In [0]:
all_columns = dim_language.columns
display(all_columns)

In [0]:
# define the primary key
primary_key = ['language']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    print(additional_columns)

In [0]:
handle_table_update(
    dim_language,
    target_dim_language,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode  # Use "update" for update mode, "full" for delete/insert mode
    )